In [ ]:
import torch
import cv2
from ultralytics import YOLO
from flask import Flask, send_file, jsonify, request
import os
import json
import mysql.connector
import socket
import threading

app = Flask(__name__)

# YOLO 모델 로드
model = YOLO("best.pt")

# Motion MJPEG 스트리밍 주소
stream_url = "http://192.168.171.2:8081/?action=stream"  # 그때그때 주소 바꿀 것

# MySQL DB 설정
db_config = {
    "host": "127.0.0.1",
    "port": 3306,
    "user": "root",
    "password": "1234",
    "database": "pickandcookdb"
}

# 영어 라벨을 한글로 변환하는 매핑
label_translation = {
    "chicken": "닭고기",
    "beef": "소고기",
    "fork": "돼지고기",
    "green onion": "대파",
    "onion": "양파",
    "photato": "감자",
    "egg": "달걀",
    "garlic": "마늘"
}

# DB에 crop된 이미지를 저장하는 함수
def save_crop_to_db(user_id, label, img_bytes):
    conn = mysql.connector.connect(**db_config)
    cursor = conn.cursor()

    cursor.execute("SELECT COUNT(*) FROM fridge WHERE fridge_ingredient = %s AND user_id = %s", (label, user_id))
    exists = cursor.fetchone()[0]

    if exists:
        cursor.execute("UPDATE fridge SET photo = %s WHERE fridge_ingredient = %s AND user_id = %s", (img_bytes, label, user_id))
    else:
        cursor.execute("INSERT INTO fridge (user_id, fridge_ingredient, photo) VALUES (%s, %s, %s)", (user_id, label, img_bytes))

    conn.commit()
    conn.close()

# DB 업데이트 함수 (라벨 기준 추가/삭제)
def update_fridge_database(user_id, detected_labels):
    conn = mysql.connector.connect(**db_config)
    cursor = conn.cursor()

    cursor.execute("SELECT fridge_ingredient FROM fridge WHERE user_id = %s", (user_id,))
    db_ingredients = set(row[0] for row in cursor.fetchall())

    to_insert = set(detected_labels) - db_ingredients
    for ingredient in to_insert:
        cursor.execute("INSERT INTO fridge (user_id, fridge_ingredient) VALUES (%s, %s)", (user_id, ingredient))

    to_delete = db_ingredients - set(detected_labels)
    for ingredient in to_delete:
        cursor.execute("DELETE FROM fridge WHERE fridge_ingredient = %s AND user_id = %s", (ingredient, user_id))

    conn.commit()
    conn.close()

# YOLO 감지를 수행하는 메인 함수
def yolo_task(user_id):
    cap = cv2.VideoCapture(stream_url)
    ret, frame = cap.read()

    if not ret:
        cap.release()
        print("카메라에서 프레임을 가져올 수 없습니다.")
        return

    results = model(frame)
    detected_objects = []
    detected_labels = []

    for result in results:
        boxes = result.boxes
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            class_id = int(box.cls)
            confidence = float(box.conf)
            label = model.names[class_id]

            translated_label = label_translation.get(label, label)
            detected_labels.append(translated_label)

            cropped = frame[y1:y2, x1:x2]
            is_success, buffer = cv2.imencode(".jpg", cropped)
            if is_success:
                img_bytes = buffer.tobytes()
                save_crop_to_db(user_id, translated_label, img_bytes)
            else:
                print(f"Failed to encode cropped image for {label}")

            detected_objects.append({
                "label": translated_label,
                "confidence": confidence,
                "bbox": [x1, y1, x2, y2]
            })

    result_img_path = "detected_frame.jpg"
    cv2.imwrite(result_img_path, frame)

    with open("detected_objects.json", "w", encoding='utf-8') as f:
        json.dump(detected_objects, f, ensure_ascii=False, indent=4)

    cap.release()

    update_fridge_database(user_id, detected_labels)

# 메인 API: 객체 인식을 트리거
@app.route('/run-yolo', methods=['POST'])
def run_yolo():
    user_id = request.form.get('userId')
    if not user_id:
        return jsonify({"error": "userId가 필요합니다."}), 400

    yolo_task(user_id)
    return jsonify({"status": "YOLO completed"})

# 서버 IP 가져오기
@app.route('/get-server-ip', methods=['GET'])
def get_server_ip():
    hostname = socket.gethostname()
    local_ip = socket.gethostbyname(hostname)
    port = request.host.split(':')[-1]
    full_address = f"http://{local_ip}:{port}"
    return jsonify({"server_ip": full_address})

# 인식된 객체 JSON 결과 가져오기
@app.route('/json', methods=['GET'])
def get_detection_json():
    if os.path.exists("detected_objects.json"):
        with open("detected_objects.json", "r", encoding='utf-8') as f:
            data = json.load(f)
        return jsonify(data)
    else:
        return "JSON 파일이 없습니다.", 404

if __name__ == '__main__':
    app.run(host='192.168.171.243', port=5000)


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://192.168.171.243:5000
Press CTRL+C to quit
